# Smart Workout Week 1 Complete Notebook
## Detailed Final Product with Eight Source Tables and Gym DB Warehouse Extension

Final-product-ready notebook for **Earth and Non** Week 1 tasks in `TODO.md`, updated for eight raw sources: four standalone Week 1 CSV datasets plus four linked tables from the **Gym Database Management System**.

This version is designed to be both an analysis notebook and a project handoff artifact. It keeps the detailed Week 1 cleaning flow for Sleep, Gym Members, ExerciseDB, and Nutrition, then adds a Gym DB warehouse layer built from users, check-ins, subscription plans, and gym locations.

## Final product purpose

- Clean and validate all eight original raw datasets using the same detailed proof style from the backup notebook.
- Add Gym DB raw inspection for four linked tables, table cleaning proof, relationship validation, and warehouse-ready summary marts.
- Create processed CSV files that can be used by backend services, Week 2 ML, Week 3 dashboard, and future recommendation logic.
- Create dashboard-ready summary tables and PNG figures.
- Document every Week 1 data decision with visible evidence.
- Produce a final checklist that maps outputs back to `TODO.md` and notes the Gym DB extension for post-Week-1 development.

## Week 1 TODO Coverage

| TODO.md line | Requirement | Evidence produced in this notebook |
|---|---|---|
| 691 + extension | Load and inspect the original 4 CSVs plus 4 Gym DB tables | Raw file audit, raw inspection summary, dtypes, missing values, duplicate counts |
| 692 | Clean Sleep dataset | Blood pressure split proof, BMI normalization proof, cleaned readiness table |
| 693 | Clean Gym Members dataset | Missing validation, BPM-response ratio, intensity band distribution |
| 694 | Clean ExerciseDB | Parsed list fields, parse proof table, coverage tables |
| 695 | Clean Nutrition dataset | Repair parser proof, numeric macro validation, macro summary |
| 696 | Derive Training Readiness | Readiness scoring table and readiness distribution |
| 697 | Outlier check on Calories_Burned | 3-sigma threshold table, outlier rows, capped calorie field |
| 700 | K-Means clustering | k=3 and k=4 silhouette table, selected k, cluster profile |
| 701 | Cross-dataset EDA | Workout, calorie, exercise coverage, nutrition, readiness, cluster plots |
| 702 | Save EDA plots as PNG | Figure manifest in `docs/figures/week1` |
| extension | Gym Database Management System | Raw table audit, relationship validation, warehouse marts, operational plots |

The TODO was originally written for four standalone CSV datasets. This notebook now covers those four CSV datasets plus four linked Gym DB tables, giving eight raw sources for the Week 1 handoff and post-Week-1 warehouse/dashboard work.

## Cell 1: Imports and Display Settings

This cell loads the libraries used for data cleaning, validation, clustering, and plotting.

The code comments are aligned so that each important line explains its role directly.
        

In [ ]:
from pathlib import Path                                                                                # Path objects keep Windows paths stable across raw, processed, notebook, and figure folders.
import ast                                                                                              # Safe parsing converts ExerciseDB string-list fields into real Python lists.
import csv                                                                                              # The nutrition file needs low-level CSV reading because some food names contain unquoted commas.
import json                                                                                             # JSON output stores manifests and dashboard-ready summary tables.
import re                                                                                               # Regular expressions standardize raw column names into snake_case.
import warnings                                                                                         # Warning control keeps notebook output focused on validation evidence.

from IPython.display import display                                                                     # Notebook display renders tables clearly in VSCode and Jupyter.
import matplotlib.pyplot as plt                                                                         # Matplotlib creates and saves static PNG figures for dashboard and report usage.
import numpy as np                                                                                      # NumPy supports vectorized scoring, thresholds, clipping, and numeric rules.
import pandas as pd                                                                                     # Pandas loads, cleans, validates, aggregates, and exports tabular datasets.
import seaborn as sns                                                                                   # Seaborn creates clean descriptive analytics plots with consistent styling.
from sklearn.cluster import KMeans                                                                      # K-Means creates sleep-based lifestyle archetypes.
from sklearn.metrics import silhouette_score                                                            # Silhouette score compares k=3 and k=4 cluster quality.
from sklearn.preprocessing import StandardScaler                                                        # StandardScaler prevents high-scale variables from dominating cluster distance.

warnings.filterwarnings("ignore", category=FutureWarning)                                               # Future warnings do not affect current Week 1 analysis results.
pd.set_option("display.max_columns", 120)                                                               # Wide table display keeps validation columns visible.
pd.set_option("display.width", 180)                                                                     # Wide text output improves readability in VSCode.
sns.set_theme(style="whitegrid", palette="Set2")                                                        # A consistent plot theme keeps Week 1 figures presentation-ready.


## Cell 2: Project Paths and Required File Contract

This cell defines the expected project root and output locations.

The notebook should be placed in:

```text
notebooks/week1/smart_workout_week1_complete_final_product.ipynb
```

It reads raw data from `data/raw`, writes cleaned files to `data/processed`, and saves figures to `docs/figures/week1`.
        

In [ ]:
RAW_FILES = {                                                                                                          # The required original raw dataset filenames are stored in one contract dictionary.
    "sleep": "Sleep_health_and_lifestyle_dataset.csv",                                                                # Sleep dataset supports readiness and lifestyle clustering.
    "gym": "gym_members_exercise_tracking.csv",                                                                       # Gym dataset supports calorie and intensity modeling.
    "exercise": "exercisedb_all_raw_flat.csv",                                                                        # ExerciseDB supports recommendation and RAG instructions.
    "nutrition": "daily_food_nutrition_dataset.csv",                                                                  # Nutrition dataset supports macro dashboard context.
}                                                                                                                      # End of original raw-file contract.

GYM_DB_DIR_NAME = "Gym Database Management System"                                                                     # The new operational dataset lives inside data/raw as a folder.
GYM_DB_FILES = {                                                                                                       # Gym DB tables are stored as a relational-style file contract.
    "users": "users_data.csv",                                                                                        # Users table behaves like a member dimension table.
    "subscription_plans": "subscription_plans.csv",                                                                    # Subscription plans table behaves like a plan dimension table.
    "gym_locations": "gym_locations_data.csv",                                                                        # Gym locations table behaves like a branch dimension table.
    "checkins": "checkin_checkout_history_updated.csv",                                                               # Check-in history behaves like the central workout-session fact table.
}                                                                                                                      # End of Gym DB file contract.
SAVE_FULL_GYMDB_FACT = False                                                                                          # The 300k-row enriched fact table stays in memory unless a full export is intentionally needed.

def find_project_root(start: Path) -> Path:                                                                             # Locate the repository root without using any computer-specific absolute path.
    candidates = [start.resolve(), *start.resolve().parents]                                                            # Search from the current working folder upward through all parent folders.
    for candidate in candidates:                                                                                        # Each candidate folder is tested as a possible project root.
        raw_dir = candidate / "data" / "raw"                                                                            # A valid Smart Workout root must contain the raw data folder.
        has_required_files = raw_dir.exists() and all((raw_dir / filename).exists() for filename in RAW_FILES.values()) # All eight original raw CSV files must exist before the folder is accepted.
        has_gym_db_folder = (raw_dir / GYM_DB_DIR_NAME).exists()                                                        # The new Gym DB folder is required for this integrated notebook.
        if has_required_files and has_gym_db_folder:                                                                    # The first matching folder is treated as the project root.
            return candidate                                                                                            # Returning this folder keeps outputs inside the cloned repository.
    raise FileNotFoundError(                                                                                            # A clear error is raised when the notebook is opened outside the project tree.
        "Could not locate the Smart Workout project root. Open VSCode/Jupyter from the repository folder that contains data/raw and the Gym DB folder."
    )                                                                                                                   # End of project-root error message.

PROJECT_ROOT = find_project_root(Path.cwd())                                                                            # The project root is discovered from the current working directory.
RAW_DIR = PROJECT_ROOT / "data" / "raw"                                                                                 # Raw CSV files remain unchanged in this folder.
GYM_DB_DIR = RAW_DIR / GYM_DB_DIR_NAME                                                                                  # Gym DB source tables are loaded from this folder.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"                                                                     # Cleaned CSV files are exported here for backend and ML usage.
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures" / "week1"                                                             # EDA PNG figures are exported here for dashboard and report reuse.
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "week1"                                                                    # This folder is the recommended final location for the notebook.

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)                                                                        # The processed-data output folder is created when missing.
FIGURES_DIR.mkdir(parents=True, exist_ok=True)                                                                          # The Week 1 figure-output folder is created when missing.
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)                                                                         # The recommended notebook folder is created when missing.

print("Project root:", PROJECT_ROOT)                                                                                    # The selected project root is printed for reproducibility.
print("Raw folder:", RAW_DIR)                                                                                           # The raw data folder is printed before file loading.
print("Gym DB folder:", GYM_DB_DIR)                                                                                     # The Gym DB folder is printed before table loading.
print("Processed folder:", PROCESSED_DIR)                                                                               # The processed output folder is printed before exports.
print("Figures folder:", FIGURES_DIR)                                                                                   # The figure output folder is printed before plot saving.

assert RAW_DIR.exists(), f"Missing raw data folder: {RAW_DIR}"                                                          # The notebook stops early if the raw data folder is unavailable.
assert GYM_DB_DIR.exists(), f"Missing Gym DB folder: {GYM_DB_DIR}"                                                       # The notebook stops early if the new Gym DB folder is unavailable.


## Cell 3: Raw File Audit Table

This table proves that all eight required raw sources exist before cleaning begins.

Why this matters:

- Missing files should fail early.
- File size helps detect incomplete downloads.
- The audit table supports the Week 1 report narrative.
        

In [ ]:
file_audit_rows = []                                                                                                    # Each row stores one raw-file availability check.

for dataset_name, filename in RAW_FILES.items():                                                                        # Each original dataset is checked from the central contract.
    file_path = RAW_DIR / filename                                                                                      # The full path is built from the raw folder and filename.
    file_audit_rows.append({                                                                                            # One audit dictionary is appended per original dataset.
        "dataset_group": "original_week1",                                                                              # Dataset group separates TODO sources from the Gym DB extension.
        "dataset": dataset_name,                                                                                        # Dataset role name.
        "filename": filename,                                                                                           # Exact raw filename expected by the notebook.
        "exists": file_path.exists(),                                                                                   # Boolean file existence result.
        "size_kb": round(file_path.stat().st_size / 1024, 2) if file_path.exists() else 0,                              # File size helps detect empty or partial files.
        "path": str(file_path),                                                                                         # Full path documents the source location.
    })                                                                                                                  # End of one original audit record.

for table_name, filename in GYM_DB_FILES.items():                                                                       # Each new Gym DB table is checked from the relational file contract.
    file_path = GYM_DB_DIR / filename                                                                                   # The table path is built from the Gym DB folder and filename.
    file_audit_rows.append({                                                                                            # One audit dictionary is appended per Gym DB table.
        "dataset_group": "gym_database_management_system",                                                              # Dataset group marks the extension source.
        "dataset": table_name,                                                                                          # Table role name.
        "filename": filename,                                                                                           # Exact Gym DB raw filename expected by the notebook.
        "exists": file_path.exists(),                                                                                   # Boolean file existence result.
        "size_kb": round(file_path.stat().st_size / 1024, 2) if file_path.exists() else 0,                              # File size helps detect empty or partial table files.
        "path": str(file_path),                                                                                         # Full path documents the source location.
    })                                                                                                                  # End of one Gym DB audit record.

file_audit = pd.DataFrame(file_audit_rows)                                                                              # The audit list becomes a readable validation table.
display(file_audit)                                                                                                     # The table is displayed before any data transformation.

missing_raw_files = file_audit.loc[~file_audit["exists"], "filename"].tolist()                                          # Missing filenames are isolated for a clear failure message.
assert not missing_raw_files, f"Missing required raw files: {missing_raw_files}"                                        # All original and Gym DB files are required for the integrated notebook.


## Cell 4: Helper Functions for Inspection, Cleaning, Export, and Plot Saving

These helpers keep the notebook consistent.

They also make the final output easier to audit because every dataset follows the same inspection and export pattern.
        

In [ ]:
def clean_column_name(name: str) -> str:                                                                # Raw column names are converted into Python-friendly snake_case names.
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", str(name))                                            # camelCase names such as bodyParts become body_Parts before final cleanup.
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)                                                          # Spaces, parentheses, slashes, and symbols become underscores.
    return name.strip("_").lower()                                                                      # Lowercase snake_case prevents later feature-engineering mistakes.

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:                                              # A dataframe copy is returned with standardized column names.
    cleaned = df.copy()                                                                                 # The raw dataframe remains unchanged for proof tables.
    cleaned.columns = [clean_column_name(column) for column in cleaned.columns]                         # Every column follows the same naming rule.
    return cleaned                                                                                      # The standardized dataframe is returned for cleaning steps.

def dataset_inspection(df: pd.DataFrame, name: str) -> dict:                                            # Core inspection metrics are collected for one dataframe.
    return {                                                                                            # The dictionary becomes one row in the raw inspection summary.
        "dataset": name,                                                                                # Dataset display name.
        "rows": int(df.shape[0]),                                                                       # Number of records.
        "columns": int(df.shape[1]),                                                                    # Number of fields.
        "total_missing": int(df.isna().sum().sum()),                                                    # Total missing cells across the dataset.
        "duplicate_rows": int(df.duplicated().sum()),                                                   # Exact duplicate row count.
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 3),                          # Approximate in-memory size for documentation.
    }                                                                                                   # End of inspection dictionary.

def show_raw_inspection(df: pd.DataFrame, name: str, preview_rows: int = 5) -> dict:                    # Detailed inspection tables are displayed for one raw dataframe.
    summary = dataset_inspection(df, name)                                                              # The compact inspection summary is created first.
    print(f'=== {name}: SUMMARY ===')                                                                   # A visible section title keeps notebook output organized.
    display(pd.DataFrame([summary]))                                                                    # The compact summary is displayed as a table.
    print(f'=== {name}: DTYPES ===')                                                                    # Column dtype evidence is shown before cleaning.
    display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))                # Dtype table reveals numeric fields loaded as text.
    print(f'=== {name}: MISSING VALUES ===')                                                            # Missing-value evidence is shown before cleaning.
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))                     # Sorted missing counts highlight the biggest cleaning issues.
    print(f'=== {name}: FIRST ROWS ===')                                                                # First rows show headers, units, and visible raw structure.
    display(df.head(preview_rows))                                                                      # Preview rows help verify file parsing.
    print(f'=== {name}: LAST ROWS ===')                                                                 # Last rows reveal footer notes or trailing irregular records.
    display(df.tail(preview_rows))                                                                      # Tail preview helps confirm file ending quality.
    return summary                                                                                      # The summary is returned for the final raw inspection table.

def parse_string_list(value) -> list[str]:                                                              # ExerciseDB string-list fields are converted into normalized lists.
    if pd.isna(value):                                                                                  # Missing values become empty lists instead of text nan values.
        return []                                                                                       # Empty lists are easier to filter and explode later.
    if isinstance(value, list):                                                                         # Already parsed lists can appear after rerunning cells.
        return [str(item).strip().lower() for item in value if str(item).strip()]                       # Existing list items are normalized.
    try:                                                                                                # Safe parsing is attempted first.
        parsed = ast.literal_eval(str(value))                                                           # literal_eval parses strings such as ['back'] without executing arbitrary code.
    except (SyntaxError, ValueError):                                                                   # Malformed values are preserved instead of stopping the notebook.
        parsed = [value]                                                                                # The raw value becomes one list item for later review.
    if isinstance(parsed, list):                                                                        # Parsed lists are normalized item by item.
        return [str(item).strip().lower() for item in parsed if str(item).strip()]                      # Whitespace and capitalization are cleaned.
    return [str(parsed).strip().lower()] if str(parsed).strip() else []                                 # Scalar parsed values are wrapped as a list.

def count_list_values(series: pd.Series) -> pd.Series:                                                  # List-column category values are counted after exploding.
    return series.explode().dropna().value_counts()                                                     # Exploding turns each list item into a separate countable row.

def export_dataframe(df: pd.DataFrame, filename: str) -> Path:                                          # Processed tables are saved in data/processed.
    output_path = PROCESSED_DIR / filename                                                              # The output path is created from the shared processed folder.
    df.to_csv(output_path, index=False, encoding="utf-8")                                               # CSV export excludes artificial notebook indexes.
    print(f'Saved processed file: {output_path}')                                                       # The exact saved path is printed for traceability.
    return output_path                                                                                  # The path is returned for manifest construction.

def save_current_plot(filename: str) -> Path:                                                           # Active matplotlib figures are saved as Week 1 PNG assets.
    output_path = FIGURES_DIR / filename                                                                # All plots are stored under docs/figures/week1.
    plt.tight_layout()                                                                                  # Layout tightening reduces clipped labels.
    plt.savefig(output_path, dpi=160, bbox_inches="tight")                                              # 160 DPI is clear enough for dashboard, report, and slide usage.
    plt.show()                                                                                          # The plot is rendered inside the notebook for visual review.
    plt.close()                                                                                         # The figure is closed to avoid overlap with later plots.
    return output_path                                                                                  # The saved path is returned for figure manifest construction.


## Cell 5: Load Raw Datasets and Build Raw Inspection Summary

This cell loads the four standalone Week 1 CSV datasets and produces raw inspection evidence.

For nutrition, a repair-aware loader is introduced because the raw file contains unquoted commas inside several `Food_Item` values.
        

In [ ]:
sleep_raw = pd.read_csv(RAW_DIR / RAW_FILES['sleep'])                                                   # Sleep CSV loads directly with pandas.
gym_raw = pd.read_csv(RAW_DIR / RAW_FILES['gym'])                                                       # Gym tracking CSV loads directly with pandas.
exercise_raw = pd.read_csv(RAW_DIR / RAW_FILES['exercise'])                                             # ExerciseDB CSV loads directly, but list-like fields still need parsing.

nutrition_path = RAW_DIR / RAW_FILES['nutrition']                                                       # Nutrition path is stored because the file needs repair-aware parsing.
nutrition_default_error = None                                                                          # The default parser error is stored for data-quality proof.
try:                                                                                                    # A default pandas read is attempted to document the parsing issue.
    nutrition_default_attempt = pd.read_csv(nutrition_path)                                             # This may fail because of unquoted commas inside food names.
except Exception as error:                                                                              # Parser failure is captured instead of stopping the notebook.
    nutrition_default_error = f'{type(error).__name__}: {error}'                                        # The error message becomes evidence for the cleaning section.

def load_nutrition_csv_repaired(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:                       # The nutrition loader repairs rows with extra comma splits in Food_Item.
    with path.open("r", encoding="utf-8-sig", newline="") as handle:                                    # BOM-safe UTF-8 keeps the header clean.
        reader = csv.reader(handle)                                                                     # csv.reader exposes raw row field counts.
        header = next(reader)                                                                           # The header defines the expected logical field count.
        expected_count = len(header)                                                                    # Expected count is used to detect malformed rows.
        repaired_rows = []                                                                              # Repaired logical rows are collected here.
        repair_log = []                                                                                 # Repair evidence is collected row by row.
        for line_number, row in enumerate(reader, start=2):                                             # Line numbers start at 2 because line 1 is the header.
            raw_count = len(row)                                                                        # Raw field count shows whether the row structure is valid.
            if raw_count == expected_count:                                                             # Rows with the expected count are already valid.
                fixed_row = row                                                                         # The original row is preserved.
                repair_action = 'no_repair_needed'                                                      # Repair log records that no change was needed.
            elif raw_count > expected_count:                                                            # Extra fields usually indicate commas inside Food_Item.
                extra_fields = raw_count - expected_count                                               # The number of extra fields tells how many initial fragments to rejoin.
                fixed_food_item = ','.join(row[: extra_fields + 1])                                     # Food item fragments are joined back together.
                fixed_row = [fixed_food_item] + row[extra_fields + 1 :]                                 # The repaired food item is combined with the remaining columns.
                repair_action = 'food_item_comma_repaired'                                              # Repair log records an actual row repair.
            else:                                                                                       # Short rows are padded to preserve rectangular structure.
                fixed_row = row + [''] * (expected_count - raw_count)                                   # Blank values preserve missing trailing fields.
                repair_action = 'short_row_padded'                                                      # Repair log records a short-row issue.
            repaired_rows.append(fixed_row)                                                             # The repaired row is stored for dataframe creation.
            repair_log.append({                                                                         # A row-level repair record is stored for validation proof.
                'line_number': line_number,                                                             # Original CSV line number.
                'raw_field_count': raw_count,                                                           # Number of raw fields before repair.
                'expected_field_count': expected_count,                                                 # Expected logical column count.
                'repair_action': repair_action,                                                         # Repair action taken for this row.
                'food_item_after_repair': fixed_row[0],                                                 # Food item after possible repair.
            })                                                                                          # End of one repair-log record.
    return pd.DataFrame(repaired_rows, columns=header), pd.DataFrame(repair_log)                        # The repaired dataframe and proof log are returned.

nutrition_raw, nutrition_repair_log = load_nutrition_csv_repaired(nutrition_path)                       # Nutrition data is loaded with the repair-aware parser.

raw_summaries = []                                                                                      # Raw inspection summaries are collected across the four standalone Week 1 datasets.
raw_summaries.append(show_raw_inspection(sleep_raw, 'Sleep Health Raw'))                                # Sleep raw structure is inspected.
raw_summaries.append(show_raw_inspection(gym_raw, 'Gym Members Raw'))                                   # Gym raw structure is inspected.
raw_summaries.append(show_raw_inspection(exercise_raw, 'ExerciseDB Raw'))                               # ExerciseDB raw structure is inspected.
raw_summaries.append(show_raw_inspection(nutrition_raw, 'Nutrition Raw Repaired'))                      # Repaired nutrition raw structure is inspected.

raw_inspection_summary = pd.DataFrame(raw_summaries)                                                    # All raw inspection summaries are combined into one table.
display(raw_inspection_summary)                                                                         # The combined raw inspection table directly satisfies TODO raw inspection evidence.

raw_inspection_output = export_dataframe(raw_inspection_summary, 'week1_raw_inspection_summary.csv')    # Raw inspection summary is exported for report evidence.


## Cell 6: Nutrition Repair Proof Table

This cell proves that the nutrition parser repaired real CSV issues rather than applying unnecessary transformation.

Expected evidence:

- Default parser error if present.
- Field-count distribution.
- Rows with `food_item_comma_repaired`.
        

In [ ]:
print('Default nutrition parser error:', nutrition_default_error)                                       # The default parser issue is documented when pandas cannot read the file normally.

nutrition_field_count_summary = nutrition_repair_log['raw_field_count'].value_counts().sort_index().reset_index() # Raw field-count distribution quantifies the CSV issue.
nutrition_field_count_summary.columns = ['raw_field_count', 'row_count']                                # Column names make the proof table readable.
display(nutrition_field_count_summary)                                                                  # Rows with more fields than expected indicate unquoted comma problems.

nutrition_repaired_rows = nutrition_repair_log[nutrition_repair_log['repair_action'] != 'no_repair_needed'].copy() # Only rows that needed repair are isolated for proof.
display(nutrition_repaired_rows.head(20))                                                               # Repaired examples show exactly which food names had comma issues.

nutrition_repair_summary_output = export_dataframe(nutrition_repair_log, 'nutrition_csv_repair_log.csv') # The full repair log is exported as final-product evidence.


## Cell 7: Load and Inspect Gym Database Management System Raw Tables

This cell loads the four Gym DB tables after the standalone Week 1 CSV datasets have already been inspected.

Why this inspection is needed:

- the Gym DB dataset is relational: users, subscription plans, gym locations, and check-in history play different database roles.
- the check-in history table is much larger than the original Week 1 datasets, so shape and memory proof are important before later joins.
- raw inspection keeps the extension transparent without changing the original Week 1 cleaning logic.
- the combined raw inspection table becomes the single audit source for both the original TODO datasets and the new operational dataset.

In [ ]:
gymdb_raw_tables = {}                                                                                                   # Raw Gym DB tables are stored by logical table name.
gymdb_raw_summaries = []                                                                                                # Raw inspection summaries are collected for the Gym DB extension.

for table_name, filename in GYM_DB_FILES.items():                                                                       # Every Gym DB table is loaded from the file contract.
    table_path = GYM_DB_DIR / filename                                                                                  # The source path is built from the Gym DB directory.
    table_df = pd.read_csv(table_path)                                                                                  # Gym DB CSV tables load directly with pandas.
    gymdb_raw_tables[table_name] = table_df                                                                             # The raw table is stored for later cleaning and warehouse joins.
    gymdb_raw_summaries.append(show_raw_inspection(table_df, f"Gym DB {table_name} Raw", preview_rows=3))               # The same inspection helper keeps proof style consistent.

gymdb_raw_inspection_summary = pd.DataFrame(gymdb_raw_summaries)                                                        # Gym DB raw summaries become a separate extension table.
gymdb_raw_inspection_summary["dataset_group"] = "gym_database_management_system"                                        # Dataset group labels the new source in exported evidence.
raw_inspection_summary["dataset_group"] = "original_week1"                                                              # Dataset group labels the original four TODO sources.
raw_inspection_summary = pd.concat([raw_inspection_summary, gymdb_raw_inspection_summary], ignore_index=True)           # Original and new inspection summaries are combined.

display(gymdb_raw_inspection_summary)                                                                                  # The Gym DB raw summary is displayed as extension evidence.
display(raw_inspection_summary)                                                                                         # The combined raw summary confirms all source tables are visible.

raw_inspection_output = export_dataframe(raw_inspection_summary, "week1_raw_inspection_summary.csv")                    # The original raw inspection output is refreshed to include Gym DB.
gymdb_raw_inspection_output = export_dataframe(gymdb_raw_inspection_summary, "gymdb_raw_inspection_summary.csv")        # A separate Gym DB inspection file supports warehouse documentation.


## Cell 8: Clean Sleep Data and Produce Cleaning Proof

This cell addresses:

- Blood pressure split into systolic and diastolic values.
- BMI category normalization.
- Missing `Sleep Disorder` cleanup.

The proof tables show before-and-after values instead of only saving a cleaned file.
        

In [ ]:
sleep_clean = standardize_columns(sleep_raw)                                                            # Sleep columns are standardized before feature engineering.

sleep_bp_before = sleep_raw[['Blood Pressure']].head(12).copy()                                         # A before-table preserves the original combined blood pressure format.
bp_parts = sleep_clean['blood_pressure'].astype(str).str.extract(r'(?P<systolic_bp>\d+)\s*/\s*(?P<diastolic_bp>\d+)') # Regex captures both blood pressure numbers from values such as 126/83.
sleep_clean['systolic_bp'] = pd.to_numeric(bp_parts['systolic_bp'], errors='coerce')                    # Systolic blood pressure becomes numeric for safety rules.
sleep_clean['diastolic_bp'] = pd.to_numeric(bp_parts['diastolic_bp'], errors='coerce')                  # Diastolic blood pressure becomes numeric for safety rules.

sleep_clean['bmi_category_normalized'] = sleep_clean['bmi_category'].replace({'Normal Weight': 'Normal'}) # Normal Weight is merged into Normal to remove duplicate category meaning.
sleep_clean['sleep_disorder'] = sleep_clean['sleep_disorder'].fillna('None')                            # Missing disorder values become an explicit None category for descriptive analysis.

sleep_bp_after = sleep_clean[['blood_pressure', 'systolic_bp', 'diastolic_bp']].head(12).copy()         # The after-table proves that blood pressure was split correctly.
bmi_before_after = pd.DataFrame({                                                                       # BMI normalization proof compares raw and normalized category counts.
    'raw_bmi_count': sleep_clean['bmi_category'].value_counts(),                                        # Raw BMI category counts show the original labels.
    'normalized_bmi_count': sleep_clean['bmi_category_normalized'].value_counts(),                      # Normalized counts show the merged category result.
}).fillna(0).astype(int)                                                                                # Missing count combinations become zero for a clean proof table.

sleep_cleaning_proof = pd.DataFrame({                                                                   # Critical cleaning fields are checked for missing values after transformation.
    'field': ['systolic_bp', 'diastolic_bp', 'bmi_category_normalized', 'sleep_disorder'],              # Fields required by readiness and safety logic.
    'missing_after_cleaning': [                                                                         # Missing values are counted after cleaning.
        sleep_clean['systolic_bp'].isna().sum(),                                                        # Systolic missing count.
        sleep_clean['diastolic_bp'].isna().sum(),                                                       # Diastolic missing count.
        sleep_clean['bmi_category_normalized'].isna().sum(),                                            # BMI category missing count.
        sleep_clean['sleep_disorder'].isna().sum(),                                                     # Sleep disorder missing count after None fill.
    ],                                                                                                  # End of missing counts.
})                                                                                                      # End of sleep cleaning proof table.

display(sleep_bp_before)                                                                                # Original blood pressure values are displayed for before evidence.
display(sleep_bp_after)                                                                                 # Split blood pressure values are displayed for after evidence.
display(bmi_before_after)                                                                               # BMI normalization evidence is displayed.
display(sleep_cleaning_proof)                                                                           # Critical field validation evidence is displayed.

sleep_cleaning_proof_output = export_dataframe(sleep_cleaning_proof, 'sleep_cleaning_proof.csv')        # Sleep cleaning proof is exported for report evidence.


## Cell 9: Derive Training Readiness Rule

This cell directly implements the TODO item:

```text
sleep quality + duration + stress + BMI -> Low/Mid/High band
```

Scoring rule:

- Sleep duration: 2 points if 7+ hours, 1 point if 6 to 6.99 hours, 0 otherwise.
- Sleep quality: 2 points if 8+, 1 point if 6 to 7, 0 otherwise.
- Stress: 2 points if 4 or lower, 1 point if 5 to 6, 0 otherwise.
- BMI: 2 points for Normal, 1 point for Overweight, 0 for Obese.

Band rule:

- Low: 0 to 3.
- Mid: 4 to 5.
- High: 6 to 8.
        

In [ ]:
sleep_clean['sleep_duration_score'] = np.select(                                                        # Sleep duration is converted into a simple readiness contribution.
    [sleep_clean['sleep_duration'] >= 7.0, sleep_clean['sleep_duration'] >= 6.0],                       # Seven or more hours receives the strongest score.
    [2, 1],                                                                                             # Score values represent strong and partial readiness contribution.
    default=0,                                                                                          # Short sleep receives no duration contribution.
)                                                                                                       # End of sleep duration scoring.

sleep_clean['sleep_quality_score'] = np.select(                                                         # Sleep quality is converted into a readiness contribution.
    [sleep_clean['quality_of_sleep'] >= 8, sleep_clean['quality_of_sleep'] >= 6],                       # High and moderate quality thresholds are separated.
    [2, 1],                                                                                             # Score values represent strong and partial readiness contribution.
    default=0,                                                                                          # Poor sleep quality receives no quality contribution.
)                                                                                                       # End of sleep quality scoring.

sleep_clean['stress_score'] = np.select(                                                                # Stress is scored inversely because lower stress supports higher readiness.
    [sleep_clean['stress_level'] <= 4, sleep_clean['stress_level'] <= 6],                               # Low and moderate stress thresholds are separated.
    [2, 1],                                                                                             # Score values represent strong and partial readiness contribution.
    default=0,                                                                                          # High stress receives no stress contribution.
)                                                                                                       # End of stress scoring.

sleep_clean['bmi_score'] = sleep_clean['bmi_category_normalized'].str.lower().map({'normal': 2, 'overweight': 1, 'obese': 0}).fillna(1) # BMI category becomes a transparent readiness contribution.

sleep_clean['readiness_score'] = (                                                                      # The final readiness score is the sum of all component scores.
    sleep_clean['sleep_duration_score']                                                                 # Sleep duration component.
    + sleep_clean['sleep_quality_score']                                                                # Sleep quality component.
    + sleep_clean['stress_score']                                                                       # Stress component.
    + sleep_clean['bmi_score']                                                                          # BMI component.
)                                                                                                       # End of readiness score formula.

sleep_clean['readiness_band'] = np.select(                                                              # The numeric readiness score is converted into dashboard-friendly bands.
    [sleep_clean['readiness_score'] <= 3, sleep_clean['readiness_score'] <= 5],                         # Low and Mid thresholds are applied before High default.
    ['Low', 'Mid'],                                                                                     # Band labels match the TODO requirement.
    default='High',                                                                                     # Scores 6 to 8 become High readiness.
)                                                                                                       # End of readiness band assignment.

sleep_clean['elevated_bp_flag'] = (sleep_clean['systolic_bp'] >= 140) | (sleep_clean['diastolic_bp'] >= 90) # Elevated blood pressure is stored for later plan safety notes.
sleep_clean['high_heart_rate_flag'] = sleep_clean['heart_rate'] >= 90                                   # High resting heart rate is stored for later plan safety notes.

readiness_component_summary = sleep_clean.groupby('readiness_band')[                                    # Component averages explain why each band differs.
    ['sleep_duration_score', 'sleep_quality_score', 'stress_score', 'bmi_score', 'readiness_score']     # Component score columns are summarized by band.
].mean().round(2).reset_index()                                                                         # Rounded averages create a readable proof table.

readiness_distribution = sleep_clean['readiness_band'].value_counts().reindex(['Low', 'Mid', 'High']).rename_axis('readiness_band').reset_index(name='record_count') # Band counts prove that Low/Mid/High labels were created.

display(readiness_component_summary)                                                                    # Readiness score component proof is displayed.
display(readiness_distribution)                                                                         # Readiness band distribution is displayed.
display(sleep_clean[['sleep_duration', 'quality_of_sleep', 'stress_level', 'bmi_category_normalized', 'readiness_score', 'readiness_band']].head(15)) # Sample records verify the rule at row level.

readiness_component_output = export_dataframe(readiness_component_summary, 'readiness_component_summary.csv') # Readiness component proof is exported.
sleep_readiness_output = export_dataframe(sleep_clean, 'sleep_cleaned_readiness.csv')                   # Cleaned sleep readiness data is exported for downstream work.


## Cell 10: Clean Gym Data, Build Intensity Target, and Prove Outlier Handling

This cell completes the gym-data TODO items:

- Verify missing values.
- Derive `bpm_response_ratio = Avg_BPM / Max_BPM`.
- Bin intensity into Low/Mid/High using the given thresholds.
- Check `Calories_Burned` outliers using 3-sigma.
- Create capped calorie values for safer modeling.
        

In [ ]:
gym_clean = standardize_columns(gym_raw)                                                                # Gym columns are standardized before target engineering.

gym_missing_summary = gym_clean.isna().sum().rename_axis('column').reset_index(name='missing_count')    # Missing values are counted for every gym column.
display(gym_missing_summary)                                                                            # Missing-value proof satisfies the gym validation TODO item.

gym_clean['bpm_response_ratio'] = gym_clean['avg_bpm'] / gym_clean['max_bpm']                           # BPM response ratio measures average effort relative to maximum BPM.
gym_clean['intensity_band'] = pd.cut(                                                                   # The continuous BPM ratio is converted into a classification target.
    gym_clean['bpm_response_ratio'],                                                                    # The derived ratio is the input to the binning rule.
    bins=[-np.inf, 0.70, 0.85, np.inf],                                                                 # The thresholds come directly from TODO.md.
    labels=['Low', 'Mid', 'High'],                                                                      # Target labels match the final classification classes.
    include_lowest=True,                                                                                # The lowest values are included in the first bin.
)                                                                                                       # End of intensity band derivation.

gym_clean['calorie_burn_rate_kcal_per_hour'] = gym_clean['calories_burned'] / gym_clean['session_duration_hours'] # Calories per hour becomes the Week 2 regression target.

calorie_mean = gym_clean['calories_burned'].mean()                                                      # Mean is required for the 3-sigma outlier boundary.
calorie_std = gym_clean['calories_burned'].std()                                                        # Standard deviation defines expected calorie spread.
calorie_lower_bound = max(0, calorie_mean - 3 * calorie_std)                                            # The lower bound is clipped at zero because negative calories are impossible.
calorie_upper_bound = calorie_mean + 3 * calorie_std                                                    # The upper bound marks unusually high calorie burn values.
gym_clean['calories_burned_zscore'] = (gym_clean['calories_burned'] - calorie_mean) / calorie_std       # Z-score records distance from the mean for each session.
gym_clean['calorie_outlier_3sigma'] = gym_clean['calories_burned_zscore'].abs() > 3                     # Rows beyond three standard deviations are flagged.
gym_clean['calories_burned_clean'] = gym_clean['calories_burned'].clip(lower=calorie_lower_bound, upper=calorie_upper_bound) # Capping preserves row count while limiting extreme influence.

intensity_distribution = gym_clean['intensity_band'].value_counts().reindex(['Low', 'Mid', 'High']).rename_axis('intensity_band').reset_index(name='session_count') # Target-class counts validate the intensity label distribution.
outlier_proof = pd.DataFrame([{                                                                         # A one-row table documents the complete outlier rule.
    'calories_mean': round(calorie_mean, 2),                                                            # Mean used for 3-sigma calculation.
    'calories_std': round(calorie_std, 2),                                                              # Standard deviation used for 3-sigma calculation.
    'lower_bound_3sigma': round(calorie_lower_bound, 2),                                                # Lower outlier boundary.
    'upper_bound_3sigma': round(calorie_upper_bound, 2),                                                # Upper outlier boundary.
    'outlier_row_count': int(gym_clean['calorie_outlier_3sigma'].sum()),                                # Number of rows outside the 3-sigma boundary.
    'original_max_calories': float(gym_clean['calories_burned'].max()),                                 # Maximum raw calorie value.
    'cleaned_max_calories': float(gym_clean['calories_burned_clean'].max()),                            # Maximum calorie value after capping.
}])                                                                                                     # End of outlier proof table.
outlier_rows = gym_clean[gym_clean['calorie_outlier_3sigma']].copy()                                    # Actual outlier rows are isolated for review.

display(intensity_distribution)                                                                         # Intensity target distribution is displayed.
display(outlier_proof)                                                                                  # 3-sigma proof table is displayed.
display(outlier_rows[['age', 'gender', 'workout_type', 'session_duration_hours', 'calories_burned', 'calories_burned_clean', 'calories_burned_zscore']]) # Actual outlier rows are displayed for transparent cleaning evidence.

gym_output = export_dataframe(gym_clean, 'gym_cleaned_intensity_calories.csv')                          # Cleaned gym data is exported for Week 2 model training.
outlier_proof_output = export_dataframe(outlier_proof, 'gym_calorie_outlier_proof.csv')                 # Outlier proof is exported for report evidence.


## Cell 11: Clean ExerciseDB and Prove List Parsing

This cell completes the ExerciseDB TODO item.

The proof table shows raw string-list values next to parsed list values so the transformation is visible.
        

In [ ]:
exercise_clean = standardize_columns(exercise_raw)                                                      # ExerciseDB columns are standardized before parsing list-like values.
exercise_raw_standardized = standardize_columns(exercise_raw)                                           # A standardized raw copy is kept for parse-proof comparison.
exercise_list_columns = ['body_parts', 'equipments', 'target_muscles', 'secondary_muscles', 'instructions'] # These columns contain Python-list strings in the raw file.

for column in exercise_list_columns:                                                                    # Each list-like column is parsed with the same safe function.
    exercise_clean[column] = exercise_clean[column].apply(parse_string_list)                            # Parsed lists support filtering, exploding, counting, and RAG preparation.

exercise_clean['primary_body_part'] = exercise_clean['body_parts'].apply(lambda values: values[0] if values else 'unknown') # Primary body part supports fast dashboard filtering.
exercise_clean['primary_equipment'] = exercise_clean['equipments'].apply(lambda values: values[0] if values else 'unknown') # Primary equipment supports equipment-based recommendation filters.
exercise_clean['primary_target_muscle'] = exercise_clean['target_muscles'].apply(lambda values: values[0] if values else 'unknown') # Primary target muscle supports simple recommender ranking.
exercise_clean['instruction_text'] = exercise_clean['instructions'].apply(lambda values: ' '.join(values)) # Joined instruction text becomes a retrieval document for future RAG work.
exercise_clean['instruction_step_count'] = exercise_clean['instructions'].apply(len)                    # Instruction count helps check whether exercise guidance is present.

exercise_parse_proof = pd.DataFrame({                                                                   # A before-and-after parsing proof table is created.
    'name': exercise_raw_standardized['name'].head(8),                                                  # Exercise names identify the sample rows.
    'raw_body_parts': exercise_raw_standardized['body_parts'].head(8),                                  # Raw body part string values.
    'parsed_body_parts': exercise_clean['body_parts'].head(8),                                          # Parsed body part lists.
    'raw_equipments': exercise_raw_standardized['equipments'].head(8),                                  # Raw equipment string values.
    'parsed_equipments': exercise_clean['equipments'].head(8),                                          # Parsed equipment lists.
    'raw_target_muscles': exercise_raw_standardized['target_muscles'].head(8),                          # Raw target muscle string values.
    'parsed_target_muscles': exercise_clean['target_muscles'].head(8),                                  # Parsed target muscle lists.
})                                                                                                      # End of parse proof table.

body_part_coverage = count_list_values(exercise_clean['body_parts']).rename_axis('body_part').reset_index(name='exercise_count') # Body part coverage table supports dashboard EDA.
equipment_coverage = count_list_values(exercise_clean['equipments']).rename_axis('equipment').reset_index(name='exercise_count') # Equipment coverage table supports recommender feasibility analysis.
target_muscle_coverage = count_list_values(exercise_clean['target_muscles']).rename_axis('target_muscle').reset_index(name='exercise_count') # Target muscle coverage supports more detailed exercise search.

display(exercise_parse_proof)                                                                           # Raw versus parsed list proof is displayed.
display(body_part_coverage.head(15))                                                                    # Top body part coverage is displayed.
display(equipment_coverage.head(15))                                                                    # Top equipment coverage is displayed.
display(target_muscle_coverage.head(15))                                                                # Top target muscle coverage is displayed.

exercise_export = exercise_clean.copy()                                                                 # A separate export copy preserves in-memory list values for analysis.
for column in exercise_list_columns:                                                                    # CSV export needs list columns serialized as JSON strings.
    exercise_export[column] = exercise_export[column].apply(json.dumps)                                 # JSON strings can be parsed back reliably by backend code.

exercise_output = export_dataframe(exercise_export, 'exercisedb_cleaned_catalog.csv')                   # Cleaned ExerciseDB catalogue is exported for recommender and RAG work.
exercise_parse_proof_output = export_dataframe(exercise_parse_proof, 'exercisedb_parse_proof.csv')      # ExerciseDB parse proof is exported for report evidence.


## Cell 12: Clean Nutrition Macros and Produce Summary Tables

This cell completes the nutrition TODO item.

It verifies numeric macro conversion and creates meal-type summary tables for the dashboard.
        

In [ ]:
nutrition_clean = standardize_columns(nutrition_raw)                                                    # Nutrition columns are standardized after repair parsing.
numeric_nutrition_columns = [                                                                           # These fields must be numeric for macro analysis.
    'calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g',                                           # Core energy and macro fields.
    'fiber_g', 'sugars_g', 'sodium_mg', 'cholesterol_mg', 'water_intake_ml',                            # Additional nutrition fields used for context.
]                                                                                                       # End of numeric nutrition column list.

for column in numeric_nutrition_columns:                                                                # Each nutrition numeric column is converted consistently.
    nutrition_clean[column] = pd.to_numeric(nutrition_clean[column], errors='coerce')                   # Invalid numeric values become NaN for validation visibility.

nutrition_clean['food_item'] = nutrition_clean['food_item'].str.strip()                                 # Food names are trimmed after repair parsing.
nutrition_clean['category'] = nutrition_clean['category'].str.strip()                                   # Category labels are trimmed for grouping consistency.
nutrition_clean['meal_type'] = nutrition_clean['meal_type'].str.strip().str.title()                     # Meal type labels become consistent title case.
nutrition_clean['macro_total_g'] = nutrition_clean[['protein_g', 'carbohydrates_g', 'fat_g']].sum(axis=1) # Total macro grams support share calculations.
nutrition_clean['protein_share'] = np.where(nutrition_clean['macro_total_g'] > 0, nutrition_clean['protein_g'] / nutrition_clean['macro_total_g'], 0) # Protein share supports macro-mix interpretation.
nutrition_clean['carbohydrate_share'] = np.where(nutrition_clean['macro_total_g'] > 0, nutrition_clean['carbohydrates_g'] / nutrition_clean['macro_total_g'], 0) # Carbohydrate share supports macro-mix interpretation.
nutrition_clean['fat_share'] = np.where(nutrition_clean['macro_total_g'] > 0, nutrition_clean['fat_g'] / nutrition_clean['macro_total_g'], 0) # Fat share supports macro-mix interpretation.
nutrition_clean['estimated_macro_calories'] = nutrition_clean['protein_g'] * 4 + nutrition_clean['carbohydrates_g'] * 4 + nutrition_clean['fat_g'] * 9 # Macro-derived calories provide a simple consistency check.
nutrition_clean['macro_calorie_gap'] = nutrition_clean['calories_kcal'] - nutrition_clean['estimated_macro_calories'] # Reported calories minus macro-derived calories reveals source-data gaps or rounding effects.

nutrition_numeric_validation = nutrition_clean[numeric_nutrition_columns].isna().sum().rename_axis('column').reset_index(name='missing_after_numeric_conversion') # Numeric conversion proof checks all macro fields.
meal_macro_summary = nutrition_clean.groupby('meal_type')[['calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g', 'water_intake_ml']].sum().round(2).reset_index() # Macro totals by meal type support dashboard charts.
meal_macro_average = nutrition_clean.groupby('meal_type')[['calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g']].mean().round(2).reset_index() # Macro averages by meal type support report interpretation.

display(nutrition_numeric_validation)                                                                   # Numeric conversion validation is displayed.
display(meal_macro_summary)                                                                             # Meal-level macro totals are displayed.
display(meal_macro_average)                                                                             # Meal-level macro averages are displayed.
display(nutrition_clean[['food_item', 'meal_type', 'calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g', 'macro_total_g']].head(12)) # Sample cleaned nutrition rows are displayed.

nutrition_output = export_dataframe(nutrition_clean, 'nutrition_cleaned_macros.csv')                    # Cleaned nutrition macro data is exported for dashboard use.
nutrition_validation_output = export_dataframe(nutrition_numeric_validation, 'nutrition_numeric_validation.csv') # Nutrition validation proof is exported.


## Cell 13: K-Means Clustering with Silhouette Proof and Cluster Profiles

This cell completes the K-Means TODO item.

Evidence produced:

- k=3 and k=4 silhouette comparison.
- Selected k.
- Cluster profile table.
- Cluster readiness mix table.
- Suggested archetype labels for dashboard storytelling.
        

In [ ]:
cluster_feature_columns = [                                                                             # Numeric lifestyle features are selected for K-Means clustering.
    'age', 'sleep_duration', 'quality_of_sleep', 'physical_activity_level',                             # Demographic, sleep, and activity features.
    'stress_level', 'heart_rate', 'daily_steps', 'systolic_bp', 'diastolic_bp',                         # Stress, vitals, movement, and blood pressure features.
]                                                                                                       # End of cluster feature list.

cluster_input = sleep_clean[cluster_feature_columns + ['bmi_category_normalized']].copy()               # BMI category is included as an additional lifestyle signal.
cluster_input = pd.get_dummies(cluster_input, columns=['bmi_category_normalized'], dtype=float)         # Categorical BMI values are converted into numeric dummy variables.
cluster_input = cluster_input.fillna(cluster_input.median(numeric_only=True))                           # Median filling prevents clustering errors from missing values.
cluster_matrix = StandardScaler().fit_transform(cluster_input)                                          # Scaling gives each feature comparable influence in K-Means distance.

cluster_score_rows = []                                                                                 # Silhouette results are collected for each candidate k.
cluster_labels_by_k = {}                                                                                # Cluster labels are stored so the selected k can be applied later.
for k in [3, 4]:                                                                                        # TODO.md requests trying 3 to 4 user archetypes.
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)                                           # Reproducible K-Means model is created for one k value.
    labels = kmeans.fit_predict(cluster_matrix)                                                         # Each sleep record receives a cluster assignment.
    score = silhouette_score(cluster_matrix, labels)                                                    # Silhouette score measures cluster separation and cohesion.
    cluster_score_rows.append({'k': k, 'silhouette_score': score})                                      # The candidate score is saved for comparison.
    cluster_labels_by_k[k] = labels                                                                     # The labels are saved for possible final assignment.

silhouette_table = pd.DataFrame(cluster_score_rows).sort_values('silhouette_score', ascending=False)    # The best silhouette score appears first after sorting.
selected_k = int(silhouette_table.iloc[0]['k'])                                                         # The selected number of clusters is the k with the highest score.
sleep_clean['archetype_cluster'] = cluster_labels_by_k[selected_k]                                      # The selected cluster labels are attached to the sleep dataset.

cluster_profile = sleep_clean.groupby('archetype_cluster')[                                             # Cluster averages describe each archetype.
    ['age', 'sleep_duration', 'quality_of_sleep', 'physical_activity_level', 'stress_level',            # Main lifestyle profile fields.
     'heart_rate', 'daily_steps', 'systolic_bp', 'diastolic_bp', 'readiness_score']                     # Vital and readiness profile fields.
].mean().round(2).reset_index()                                                                         # Rounded cluster profile table is created for dashboard/report usage.
cluster_sizes = sleep_clean.groupby('archetype_cluster').size().reset_index(name='record_count')        # Cluster size is calculated separately.
cluster_profile = cluster_profile.merge(cluster_sizes, on='archetype_cluster', how='left')              # Cluster sizes are merged into the profile table.

cluster_readiness_mix = pd.crosstab(sleep_clean['archetype_cluster'], sleep_clean['readiness_band']).reset_index() # Readiness mix by cluster supports archetype interpretation.
cluster_profile['suggested_archetype_label'] = np.select(                                               # Rule-based labels make clusters easier to present.
    [                                                                                                   # Condition list starts.
        (cluster_profile['sleep_duration'] >= 7.5) & (cluster_profile['stress_level'] <= 4),            # High sleep and low stress suggest a recovered archetype.
        (cluster_profile['sleep_duration'] < 6.5) | (cluster_profile['stress_level'] >= 7),             # Low sleep or high stress suggest a strained archetype.
    ],                                                                                                  # End of condition list.
    ['Recovered / High Readiness', 'Strained / Low Recovery'],                                          # Suggested archetype names for clear reporting.
    default='Moderate / Mixed Readiness',                                                               # Default label covers middle-pattern clusters.
)                                                                                                       # End of suggested archetype label assignment.

display(silhouette_table)                                                                               # Silhouette comparison proves why selected_k was chosen.
print('Selected k:', selected_k)                                                                        # The selected k is printed clearly for the report narrative.
display(cluster_profile)                                                                                # Cluster profile table is displayed for final-product interpretation.
display(cluster_readiness_mix)                                                                          # Readiness mix by cluster is displayed.

sleep_clusters_output = export_dataframe(sleep_clean, 'sleep_cleaned_readiness_clusters.csv')           # Sleep data with readiness and cluster labels is exported.
silhouette_output = export_dataframe(silhouette_table, 'sleep_kmeans_silhouette_scores.csv')            # Silhouette comparison is exported as proof.
cluster_profile_output = export_dataframe(cluster_profile, 'sleep_cluster_profile.csv')                 # Cluster profile table is exported for dashboard/report usage.
cluster_readiness_output = export_dataframe(cluster_readiness_mix, 'sleep_cluster_readiness_mix.csv')   # Cluster readiness mix table is exported.


## Cell 14: Clean Gym DB Tables and Build Warehouse-Ready Features

This cell prepares the new Gym Database Management System tables for warehouse-style analysis.

Why this cleaning is needed:

- `users_data.csv` becomes a member dimension table with age groups and parsed signup dates.
- `subscription_plans.csv` becomes a plan dimension table with numeric monthly price.
- `gym_locations_data.csv` becomes a location dimension table with parsed facilities.
- `checkin_checkout_history_updated.csv` becomes the session fact source with duration, weekday, hour, month, and calories-per-minute features.
- these features prepare the project for dashboard metrics, backend services, and future recommendation rules after Week 1.

In [ ]:
gymdb_users_clean = standardize_columns(gymdb_raw_tables["users"])                                                       # Users table becomes the member dimension source.
gymdb_plans_clean = standardize_columns(gymdb_raw_tables["subscription_plans"])                                          # Subscription table becomes the plan dimension source.
gymdb_locations_clean = standardize_columns(gymdb_raw_tables["gym_locations"])                                           # Location table becomes the branch dimension source.
gymdb_checkins_clean = standardize_columns(gymdb_raw_tables["checkins"])                                                 # Check-in table becomes the session fact source.

gymdb_users_clean["birthdate"] = pd.to_datetime(gymdb_users_clean["birthdate"], errors="coerce")                        # Birthdate is parsed for demographic validation.
gymdb_users_clean["sign_up_date"] = pd.to_datetime(gymdb_users_clean["sign_up_date"], errors="coerce")                  # Signup date is parsed for lifecycle and cohort analysis.
gymdb_users_clean["age_group"] = pd.cut(                                                                                 # Age is converted into dashboard-friendly groups.
    gymdb_users_clean["age"],                                                                                            # Raw numeric age is the source for grouping.
    bins=[0, 24, 34, 44, 54, 64, np.inf],                                                                                # Age bands create stable dashboard categories.
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],                                                       # Human-readable labels are easier for presentation and filters.
)                                                                                                                       # End of age-group transformation.

gymdb_plans_clean["price_per_month"] = pd.to_numeric(gymdb_plans_clean["price_per_month"], errors="coerce")             # Plan price must be numeric for revenue summaries.
gymdb_locations_clean["facility_list"] = gymdb_locations_clean["facilities"].apply(                                      # Facilities are split into a list for future filtering.
    lambda value: [item.strip() for item in str(value).split(",") if item.strip()]                                       # Comma-separated text becomes clean facility tokens.
)                                                                                                                       # End of facility parsing.

gymdb_checkins_clean["checkin_time"] = pd.to_datetime(gymdb_checkins_clean["checkin_time"], errors="coerce")            # Check-in timestamp is parsed for time-series analysis.
gymdb_checkins_clean["checkout_time"] = pd.to_datetime(gymdb_checkins_clean["checkout_time"], errors="coerce")          # Checkout timestamp is parsed for duration calculation.
gymdb_checkins_clean["duration_minutes"] = (                                                                            # Session duration is derived from checkout minus check-in.
    gymdb_checkins_clean["checkout_time"] - gymdb_checkins_clean["checkin_time"]                                         # Datetime subtraction returns a timedelata-like value.
).dt.total_seconds() / 60                                                                                               # Seconds are converted into minutes for interpretability.
gymdb_checkins_clean["session_date"] = gymdb_checkins_clean["checkin_time"].dt.date                                      # Date supports daily aggregation if needed later.
gymdb_checkins_clean["session_month"] = gymdb_checkins_clean["checkin_time"].dt.to_period("M").astype(str)              # Month supports dashboard time-series summaries.
gymdb_checkins_clean["session_weekday"] = gymdb_checkins_clean["checkin_time"].dt.day_name()                            # Weekday supports operating pattern analysis.
gymdb_checkins_clean["session_hour"] = gymdb_checkins_clean["checkin_time"].dt.hour                                      # Hour supports peak-time dashboard analysis.
gymdb_checkins_clean["valid_duration"] = gymdb_checkins_clean["duration_minutes"].gt(0)                                  # Positive duration proves check-in/out order is valid.
gymdb_checkins_clean["calories_per_minute"] = (                                                                         # Calories per minute normalizes burn against session length.
    gymdb_checkins_clean["calories_burned"] / gymdb_checkins_clean["duration_minutes"]                                   # This field supports operational workout intensity comparison.
)                                                                                                                       # End of calories-per-minute feature.

gymdb_table_cleaning_proof = pd.DataFrame([                                                                             # Table-level proof documents readiness for warehouse joins.
    {                                                                                                                   # Users table validation row.
        "table": "gymdb_users_clean",                                                                                   # Cleaned table name.
        "rows": int(gymdb_users_clean.shape[0]),                                                                         # Number of user records.
        "columns": int(gymdb_users_clean.shape[1]),                                                                      # Number of cleaned user fields.
        "missing_values": int(gymdb_users_clean.isna().sum().sum()),                                                     # Total missing cells after cleaning.
        "duplicate_key_count": int(gymdb_users_clean["user_id"].duplicated().sum()),                                     # User ID should be unique.
        "status": "pass" if gymdb_users_clean["user_id"].duplicated().sum() == 0 else "review",                         # Status flags key problems.
    },                                                                                                                  # End of users validation row.
    {                                                                                                                   # Subscription plan validation row.
        "table": "gymdb_plans_clean",                                                                                   # Cleaned table name.
        "rows": int(gymdb_plans_clean.shape[0]),                                                                         # Number of plan records.
        "columns": int(gymdb_plans_clean.shape[1]),                                                                      # Number of plan fields.
        "missing_values": int(gymdb_plans_clean.isna().sum().sum()),                                                     # Total missing cells after cleaning.
        "duplicate_key_count": int(gymdb_plans_clean["subscription_plan"].duplicated().sum()),                           # Plan name should be unique.
        "status": "pass" if gymdb_plans_clean["subscription_plan"].duplicated().sum() == 0 else "review",                # Status flags key problems.
    },                                                                                                                  # End of plan validation row.
    {                                                                                                                   # Gym location validation row.
        "table": "gymdb_locations_clean",                                                                               # Cleaned table name.
        "rows": int(gymdb_locations_clean.shape[0]),                                                                     # Number of location records.
        "columns": int(gymdb_locations_clean.shape[1]),                                                                  # Number of location fields.
        "missing_values": int(gymdb_locations_clean.isna().sum().sum()),                                                 # Total missing cells after cleaning.
        "duplicate_key_count": int(gymdb_locations_clean["gym_id"].duplicated().sum()),                                  # Gym ID should be unique.
        "status": "pass" if gymdb_locations_clean["gym_id"].duplicated().sum() == 0 else "review",                      # Status flags key problems.
    },                                                                                                                  # End of location validation row.
    {                                                                                                                   # Check-in validation row.
        "table": "gymdb_checkins_clean",                                                                                # Cleaned table name.
        "rows": int(gymdb_checkins_clean.shape[0]),                                                                      # Number of workout-session records.
        "columns": int(gymdb_checkins_clean.shape[1]),                                                                   # Number of session fields after feature engineering.
        "missing_values": int(gymdb_checkins_clean.isna().sum().sum()),                                                  # Total missing cells after parsing dates and durations.
        "duplicate_key_count": int(gymdb_checkins_clean.duplicated().sum()),                                             # Exact duplicate session rows should be absent.
        "status": "pass" if gymdb_checkins_clean.duplicated().sum() == 0 else "review",                                 # Status flags duplicate rows.
    },                                                                                                                  # End of check-in validation row.
])                                                                                                                      # End of Gym DB cleaning proof table.

gymdb_users_output = export_dataframe(gymdb_users_clean, "gymdb_dim_users_cleaned.csv")                                 # User dimension is exported for backend/dashboard use.
gymdb_plans_output = export_dataframe(gymdb_plans_clean, "gymdb_dim_subscription_plans_cleaned.csv")                    # Plan dimension is exported for revenue summaries.
gymdb_locations_output = export_dataframe(gymdb_locations_clean, "gymdb_dim_locations_cleaned.csv")                     # Location dimension is exported for branch dashboards.
gymdb_table_cleaning_proof_output = export_dataframe(gymdb_table_cleaning_proof, "gymdb_table_cleaning_proof.csv")      # Cleaning proof is exported as report evidence.

display(gymdb_table_cleaning_proof)                                                                                     # Table-level proof is displayed before relationship validation.
display(gymdb_checkins_clean[["user_id", "gym_id", "checkin_time", "checkout_time", "duration_minutes", "calories_per_minute"]].head()) # A session-feature sample confirms derived fields.


## Cell 15: Validate Gym DB Relationships and Create Dashboard Marts

This cell builds an in-memory warehouse layer from the cleaned Gym DB tables.

Why this warehouse step is needed:

- check-in history is the fact table because each row represents one workout visit.
- users, subscription plans, and gym locations are dimension tables because they describe members, plans, and branches.
- relationship validation proves that joins are safe before dashboard summaries are created.
- aggregate marts are exported instead of a full duplicated fact table to keep the repository practical for GitHub.

In [ ]:
gymdb_relationship_validation = pd.DataFrame([                                                                         # Relationship rules prove that warehouse joins are valid.
    {                                                                                                                  # User foreign-key validation.
        "validation_rule": "Every check-in user_id exists in users",                                                   # Business-readable validation rule.
        "invalid_count": int((~gymdb_checkins_clean["user_id"].isin(gymdb_users_clean["user_id"])).sum()),             # Missing user IDs would create orphan sessions.
    },                                                                                                                 # End of user validation rule.
    {                                                                                                                  # Gym foreign-key validation.
        "validation_rule": "Every check-in gym_id exists in gym_locations",                                            # Business-readable validation rule.
        "invalid_count": int((~gymdb_checkins_clean["gym_id"].isin(gymdb_locations_clean["gym_id"])).sum()),           # Missing gym IDs would create orphan location records.
    },                                                                                                                 # End of gym validation rule.
    {                                                                                                                  # Plan foreign-key validation.
        "validation_rule": "Every user subscription_plan exists in subscription_plans",                                # Business-readable validation rule.
        "invalid_count": int((~gymdb_users_clean["subscription_plan"].isin(gymdb_plans_clean["subscription_plan"])).sum()), # Missing plans would break revenue summaries.
    },                                                                                                                 # End of plan validation rule.
    {                                                                                                                  # Duration validation.
        "validation_rule": "Every check-in has positive duration",                                                     # Business-readable validation rule.
        "invalid_count": int((~gymdb_checkins_clean["valid_duration"]).sum()),                                         # Non-positive durations would break calories-per-minute logic.
    },                                                                                                                 # End of duration validation rule.
    {                                                                                                                  # Timestamp validation.
        "validation_rule": "Every check-in has parseable checkin_time and checkout_time",                              # Business-readable validation rule.
        "invalid_count": int(gymdb_checkins_clean[["checkin_time", "checkout_time"]].isna().any(axis=1).sum()),        # Unparseable timestamps would break time-series features.
    },                                                                                                                 # End of timestamp validation rule.
])                                                                                                                     # End of relationship validation table.
gymdb_relationship_validation["status"] = np.where(gymdb_relationship_validation["invalid_count"].eq(0), "pass", "review") # A readable status is added for final reporting.

gymdb_fact_sessions = (                                                                                                # The central Gym DB fact table is created in memory.
    gymdb_checkins_clean                                                                                               # Session-level records are the fact-table base.
    .merge(                                                                                                            # Member attributes are joined from users.
        gymdb_users_clean[["user_id", "age", "age_group", "gender", "user_location", "subscription_plan", "sign_up_date"]], # Only dashboard-relevant user fields are selected.
        on="user_id",                                                                                                  # user_id links session records to members.
        how="left",                                                                                                    # Left join preserves every workout session.
    )                                                                                                                  # End of user join.
    .merge(                                                                                                            # Location attributes are joined from gyms.
        gymdb_locations_clean[["gym_id", "location", "gym_type", "facilities"]],                                       # Location fields support branch utilization.
        on="gym_id",                                                                                                   # gym_id links session records to branches.
        how="left",                                                                                                    # Left join preserves every workout session.
    )                                                                                                                  # End of location join.
    .merge(                                                                                                            # Plan price is joined from subscription plans.
        gymdb_plans_clean[["subscription_plan", "price_per_month"]],                                                   # Plan price supports revenue summary.
        on="subscription_plan",                                                                                        # subscription_plan links users to plans.
        how="left",                                                                                                    # Left join preserves every workout session.
    )                                                                                                                  # End of plan join.
)                                                                                                                      # End of in-memory fact table creation.

gymdb_fact_sample = gymdb_fact_sessions.head(1000).copy()                                                              # A small fact sample gives visible proof without exporting 300k rows.
gymdb_session_by_workout = (                                                                                           # Workout mart summarizes operational demand.
    gymdb_fact_sessions.groupby("workout_type")                                                                        # Workout type is the main operating category.
    .agg(                                                                                                              # Multiple measures support BI cards and charts.
        session_count=("user_id", "size"),                                                                             # Total sessions by workout type.
        unique_users=("user_id", "nunique"),                                                                           # Unique members participating in each workout type.
        avg_duration_minutes=("duration_minutes", "mean"),                                                             # Average session length.
        avg_calories=("calories_burned", "mean"),                                                                      # Average calories per session.
        avg_calories_per_minute=("calories_per_minute", "mean"),                                                       # Normalized intensity measure.
    )                                                                                                                  # End of aggregation definition.
    .round(3)                                                                                                          # Rounded values are easier to read in dashboards.
    .reset_index()                                                                                                     # Workout type becomes a normal column.
    .sort_values("session_count", ascending=False)                                                                     # Highest-demand workout types appear first.
)                                                                                                                      # End of workout mart.

gymdb_subscription_summary = (                                                                                         # Subscription mart summarizes plan mix and revenue.
    gymdb_users_clean.merge(gymdb_plans_clean, on="subscription_plan", how="left")                                      # Users are enriched with plan prices.
    .groupby("subscription_plan")                                                                                      # Plan is the business category.
    .agg(user_count=("user_id", "nunique"), price_per_month=("price_per_month", "first"))                              # User count and price are combined.
    .reset_index()                                                                                                     # Plan becomes a normal column.
)                                                                                                                      # End of subscription aggregation.
gymdb_subscription_summary["estimated_monthly_recurring_revenue"] = (                                                  # MRR is estimated from user count and plan price.
    gymdb_subscription_summary["user_count"] * gymdb_subscription_summary["price_per_month"]                            # The simple estimate supports BI dashboard context.
).round(2)                                                                                                             # Rounded currency-like value is easier to present.

gymdb_location_utilization = (                                                                                         # Branch mart summarizes utilization by location.
    gymdb_fact_sessions.groupby(["gym_id", "location", "gym_type"])                                                    # Location and gym type describe each branch.
    .agg(                                                                                                              # Utilization measures are collected together.
        session_count=("user_id", "size"),                                                                             # Total sessions at each branch.
        unique_users=("user_id", "nunique"),                                                                           # Unique members visiting each branch.
        avg_duration_minutes=("duration_minutes", "mean"),                                                             # Average branch session length.
    )                                                                                                                  # End of location aggregation definition.
    .round(3)                                                                                                          # Rounded values are presentation-friendly.
    .reset_index()                                                                                                     # Group keys become normal columns.
    .sort_values("session_count", ascending=False)                                                                     # Busiest branches appear first.
)                                                                                                                      # End of location mart.

gymdb_monthly_activity = (                                                                                             # Monthly mart supports trend charts.
    gymdb_fact_sessions.groupby("session_month")                                                                       # Month is derived from check-in time.
    .agg(session_count=("user_id", "size"), unique_users=("user_id", "nunique"), total_calories=("calories_burned", "sum"), avg_duration_minutes=("duration_minutes", "mean")) # Monthly BI measures.
    .round(3)                                                                                                          # Rounded values are presentation-friendly.
    .reset_index()                                                                                                     # Month becomes a normal column.
)                                                                                                                      # End of monthly mart.

gymdb_hourly_usage = (                                                                                                 # Hourly mart supports peak-time analysis.
    gymdb_fact_sessions.groupby("session_hour")                                                                        # Hour is derived from check-in time.
    .agg(session_count=("user_id", "size"), avg_calories=("calories_burned", "mean"))                                  # Hourly measures support operational planning.
    .round(3)                                                                                                          # Rounded values are presentation-friendly.
    .reset_index()                                                                                                     # Hour becomes a normal column.
)                                                                                                                      # End of hourly mart.

gymdb_gym_type_workout_summary = (                                                                                     # Gym-type mart connects workout demand to facility category.
    gymdb_fact_sessions.groupby(["gym_type", "workout_type"])                                                          # Gym type and workout type create a two-dimensional view.
    .agg(session_count=("user_id", "size"), avg_calories=("calories_burned", "mean"))                                  # Count and average calories summarize operational behavior.
    .round(3)                                                                                                          # Rounded values are presentation-friendly.
    .reset_index()                                                                                                     # Group keys become normal columns.
)                                                                                                                      # End of gym-type workout mart.

gymdb_warehouse_overview = pd.DataFrame([                                                                              # Warehouse assets are documented for handoff.
    {"asset": "gymdb_fact_sessions", "rows": int(gymdb_fact_sessions.shape[0]), "exported": SAVE_FULL_GYMDB_FACT, "purpose": "central in-memory workout-session fact table"}, # Fact-table note.
    {"asset": "gymdb_dim_users_cleaned.csv", "rows": int(gymdb_users_clean.shape[0]), "exported": True, "purpose": "member dimension"}, # User dimension note.
    {"asset": "gymdb_dim_subscription_plans_cleaned.csv", "rows": int(gymdb_plans_clean.shape[0]), "exported": True, "purpose": "subscription plan dimension"}, # Plan dimension note.
    {"asset": "gymdb_dim_locations_cleaned.csv", "rows": int(gymdb_locations_clean.shape[0]), "exported": True, "purpose": "gym branch dimension"}, # Location dimension note.
    {"asset": "gymdb_mart_session_by_workout.csv", "rows": int(gymdb_session_by_workout.shape[0]), "exported": True, "purpose": "workout demand mart"}, # Workout mart note.
    {"asset": "gymdb_mart_subscription_summary.csv", "rows": int(gymdb_subscription_summary.shape[0]), "exported": True, "purpose": "plan and revenue mart"}, # Subscription mart note.
    {"asset": "gymdb_mart_location_utilization.csv", "rows": int(gymdb_location_utilization.shape[0]), "exported": True, "purpose": "branch utilization mart"}, # Location mart note.
])                                                                                                                     # End of warehouse overview.

if SAVE_FULL_GYMDB_FACT:                                                                                               # Full fact export is intentionally optional.
    gymdb_fact_sessions_output = export_dataframe(gymdb_fact_sessions, "gymdb_fact_sessions_cleaned.csv")              # Full fact output supports backend use when repository size is acceptable.
else:                                                                                                                  # Default branch keeps the repository lightweight.
    gymdb_fact_sessions_output = None                                                                                  # No full fact CSV is created by default.

gymdb_fact_sample_output = export_dataframe(gymdb_fact_sample, "gymdb_fact_sessions_sample.csv")                       # A 1,000-row sample supports inspection and documentation.
gymdb_relationship_output = export_dataframe(gymdb_relationship_validation, "gymdb_relationship_validation.csv")       # Relationship proof is exported for report evidence.
gymdb_warehouse_output = export_dataframe(gymdb_warehouse_overview, "gymdb_warehouse_overview.csv")                    # Warehouse overview is exported for handoff.
gymdb_session_by_workout_output = export_dataframe(gymdb_session_by_workout, "gymdb_mart_session_by_workout.csv")      # Workout mart is exported.
gymdb_subscription_output = export_dataframe(gymdb_subscription_summary, "gymdb_mart_subscription_summary.csv")        # Subscription mart is exported.
gymdb_location_output = export_dataframe(gymdb_location_utilization, "gymdb_mart_location_utilization.csv")            # Location mart is exported.
gymdb_monthly_output = export_dataframe(gymdb_monthly_activity, "gymdb_mart_monthly_activity.csv")                     # Monthly mart is exported.
gymdb_hourly_output = export_dataframe(gymdb_hourly_usage, "gymdb_mart_hourly_usage.csv")                              # Hourly mart is exported.
gymdb_gym_type_output = export_dataframe(gymdb_gym_type_workout_summary, "gymdb_mart_gym_type_workout_summary.csv")   # Gym-type mart is exported.

display(gymdb_relationship_validation)                                                                                 # Relationship proof is displayed before downstream summaries.
display(gymdb_warehouse_overview)                                                                                      # Warehouse overview explains what was created.
display(gymdb_session_by_workout)                                                                                      # Workout demand mart previews operational usage.
display(gymdb_subscription_summary)                                                                                    # Subscription mart previews plan and revenue context.


## Cell 16: Cross-Dataset Summary Tables

This cell builds dashboard-ready summary tables.

These tables are useful even before the React dashboard is fully connected because they define the exact chart data shape needed later.
        

In [ ]:
workout_type_distribution = gym_clean['workout_type'].value_counts().rename_axis('workout_type').reset_index(name='session_count') # Workout type counts support the overview dashboard.
experience_calorie_summary = gym_clean.groupby('experience_level')['calories_burned_clean'].agg(['count', 'mean', 'median', 'std']).round(2).reset_index() # Calories by experience supports descriptive analytics and modeling narrative.
readiness_distribution = sleep_clean['readiness_band'].value_counts().reindex(['Low', 'Mid', 'High']).rename_axis('readiness_band').reset_index(name='record_count') # Readiness band counts support the readiness dashboard.
macro_mix_summary = meal_macro_summary.copy()                                                           # Meal macro totals are reused as the macro dashboard table.

summary_tables = {                                                                                      # All summary tables are collected into one JSON-ready object.
    'workout_type_distribution': workout_type_distribution.to_dict(orient='records'),                   # Chart-ready workout distribution.
    'intensity_distribution': intensity_distribution.to_dict(orient='records'),                         # Chart-ready intensity distribution.
    'experience_calorie_summary': experience_calorie_summary.to_dict(orient='records'),                 # Chart-ready calorie summary by experience.
    'body_part_coverage_top20': body_part_coverage.head(20).to_dict(orient='records'),                  # Top body part coverage for ExerciseDB.
    'equipment_coverage_top20': equipment_coverage.head(20).to_dict(orient='records'),                  # Top equipment coverage for ExerciseDB.
    'target_muscle_coverage_top20': target_muscle_coverage.head(20).to_dict(orient='records'),          # Top target muscle coverage for ExerciseDB.
    'meal_macro_summary': macro_mix_summary.to_dict(orient='records'),                                  # Nutrition macro mix by meal type.
    'readiness_distribution': readiness_distribution.to_dict(orient='records'),                         # Training readiness distribution.
    'cluster_profile': cluster_profile.to_dict(orient='records'),                                       # K-Means archetype profiles.
    'cluster_readiness_mix': cluster_readiness_mix.to_dict(orient='records'),                           # Readiness mix by archetype cluster.
    'gymdb_session_by_workout': gymdb_session_by_workout.to_dict(orient='records'),                     # Gym DB workout demand mart.
    'gymdb_subscription_summary': gymdb_subscription_summary.to_dict(orient='records'),                 # Gym DB subscription and MRR mart.
    'gymdb_location_utilization': gymdb_location_utilization.to_dict(orient='records'),                 # Gym DB branch utilization mart.
    'gymdb_monthly_activity': gymdb_monthly_activity.to_dict(orient='records'),                         # Gym DB monthly activity mart.
    'gymdb_hourly_usage': gymdb_hourly_usage.to_dict(orient='records'),                                 # Gym DB hourly operating pattern mart.
}                                                                                                       # End of summary table dictionary.

summary_tables_path = PROCESSED_DIR / 'week1_dashboard_summary_tables.json'                             # The summary JSON becomes a backend/dashboard handoff file.
summary_tables_path.write_text(json.dumps(summary_tables, indent=2), encoding='utf-8')                  # Readable JSON is exported for review and reuse.

display(workout_type_distribution)                                                                      # Workout distribution table is displayed.
display(experience_calorie_summary)                                                                     # Calories by experience table is displayed.
display(body_part_coverage.head(12))                                                                    # Body part coverage table is displayed.
display(equipment_coverage.head(12))                                                                    # Equipment coverage table is displayed.
display(macro_mix_summary)                                                                              # Nutrition macro summary table is displayed.
display(gymdb_session_by_workout)                                                                       # Gym DB workout demand table is displayed.
display(gymdb_location_utilization.head(10))                                                            # Gym DB branch utilization table is displayed.
display(gymdb_subscription_summary)                                                                     # Gym DB subscription summary table is displayed.
print('Saved summary JSON:', summary_tables_path)                                                       # The summary JSON path is printed for handoff.


## Cell 17: Save Final Week 1 EDA Plots as PNG

This cell saves all dashboard/report plots required for Week 1.

The figure paths are collected for the final manifest.
        

In [ ]:
figure_outputs = []                                                                                     # Saved PNG paths are collected for final validation.

plt.figure(figsize=(8, 5))                                                                              # A medium chart size works for dashboard cards and report pages.
sns.barplot(data=workout_type_distribution, x='workout_type', y='session_count')                        # Workout session counts are plotted by type.
plt.title('Workout Type Distribution')                                                                  # Chart title describes the EDA question.
plt.xlabel('Workout Type')                                                                              # x-axis label identifies the workout category.
plt.ylabel('Session Count')                                                                             # y-axis label identifies the count measure.
figure_outputs.append(save_current_plot('workout_type_distribution.png'))                               # The workout distribution plot is saved.

plt.figure(figsize=(8, 5))                                                                              # Boxplot size supports clear group comparison.
sns.boxplot(data=gym_clean, x='experience_level', y='calories_burned_clean')                            # Calories are compared across experience levels.
plt.title('Calories Burned by Experience Level')                                                        # Chart title links calories to experience.
plt.xlabel('Experience Level')                                                                          # x-axis label identifies user experience groups.
plt.ylabel('Calories Burned (3-Sigma Capped)')                                                          # y-axis label documents the cleaned calorie field.
figure_outputs.append(save_current_plot('calories_by_experience_level.png'))                            # The calorie pattern plot is saved.

plt.figure(figsize=(7, 5))                                                                              # Compact size works for three intensity classes.
sns.barplot(data=intensity_distribution, x='intensity_band', y='session_count', order=['Low', 'Mid', 'High']) # Intensity class counts are plotted in logical order.
plt.title('Workout Intensity Band Distribution')                                                        # Chart title names the derived target.
plt.xlabel('Intensity Band')                                                                            # x-axis label identifies class labels.
plt.ylabel('Session Count')                                                                             # y-axis label identifies the count measure.
figure_outputs.append(save_current_plot('intensity_band_distribution.png'))                             # The intensity target plot is saved.

plt.figure(figsize=(9, 5))                                                                              # Horizontal layout supports longer body part labels.
sns.barplot(data=body_part_coverage.head(12), x='exercise_count', y='body_part')                        # Top body part coverage is plotted.
plt.title('Exercise Coverage by Body Part')                                                             # Chart title describes catalogue coverage.
plt.xlabel('Exercise Count')                                                                            # x-axis label identifies exercise count.
plt.ylabel('Body Part')                                                                                 # y-axis label identifies body part.
figure_outputs.append(save_current_plot('exercise_coverage_by_body_part.png'))                          # The body part coverage plot is saved.

plt.figure(figsize=(9, 5))                                                                              # Horizontal layout supports longer equipment labels.
sns.barplot(data=equipment_coverage.head(12), x='exercise_count', y='equipment')                        # Top equipment coverage is plotted.
plt.title('Exercise Coverage by Equipment')                                                             # Chart title describes equipment coverage.
plt.xlabel('Exercise Count')                                                                            # x-axis label identifies exercise count.
plt.ylabel('Equipment')                                                                                 # y-axis label identifies equipment type.
figure_outputs.append(save_current_plot('exercise_coverage_by_equipment.png'))                          # The equipment coverage plot is saved.

macro_plot = macro_mix_summary.set_index('meal_type')[['protein_g', 'carbohydrates_g', 'fat_g']]        # Macro columns are selected for stacked plotting.
macro_plot.plot(kind='bar', stacked=True, figsize=(9, 5))                                               # Stacked bars show macro composition by meal type.
plt.title('Nutrition Macro Mix by Meal Type')                                                           # Chart title matches the Week 1 EDA requirement.
plt.xlabel('Meal Type')                                                                                 # x-axis label identifies meal category.
plt.ylabel('Total Grams')                                                                               # y-axis label identifies total macro grams.
figure_outputs.append(save_current_plot('nutrition_macro_mix_by_meal_type.png'))                        # The nutrition macro plot is saved.

plt.figure(figsize=(7, 5))                                                                              # Compact size works for three readiness bands.
sns.barplot(data=readiness_distribution, x='readiness_band', y='record_count', order=['Low', 'Mid', 'High']) # Readiness band counts are plotted in logical order.
plt.title('Training Readiness Distribution')                                                            # Chart title names the prescriptive readiness signal.
plt.xlabel('Readiness Band')                                                                            # x-axis label identifies readiness categories.
plt.ylabel('Record Count')                                                                              # y-axis label identifies sleep dataset records.
figure_outputs.append(save_current_plot('training_readiness_distribution.png'))                         # The readiness distribution plot is saved.

plt.figure(figsize=(8, 5))                                                                              # Scatterplot size supports cluster visualization.
sns.scatterplot(data=sleep_clean, x='sleep_duration', y='stress_level', hue='archetype_cluster', palette='Set2', s=70) # Clusters are shown using intuitive sleep and stress axes.
plt.title('Sleep Archetype Clusters')                                                                   # Chart title names the K-Means deliverable.
plt.xlabel('Sleep Duration')                                                                            # x-axis label identifies sleep amount.
plt.ylabel('Stress Level')                                                                              # y-axis label identifies stress score.
figure_outputs.append(save_current_plot('sleep_archetype_clusters.png'))                                # The cluster scatterplot is saved.

plt.figure(figsize=(9, 5))                                                                              # A medium chart size works for operational dashboard cards.
sns.barplot(data=gymdb_session_by_workout, x='workout_type', y='session_count')                         # Gym DB sessions are plotted by workout type.
plt.title('Gym DB Sessions by Workout Type')                                                            # Chart title identifies operational demand by workout category.
plt.xlabel('Workout Type')                                                                              # x-axis label identifies the workout category.
plt.ylabel('Session Count')                                                                             # y-axis label identifies total sessions.
plt.xticks(rotation=30, ha='right')                                                                     # Rotated labels prevent overlap for longer workout names.
figure_outputs.append(save_current_plot('gymdb_sessions_by_workout_type.png'))                          # The Gym DB workout demand plot is saved.

plt.figure(figsize=(9, 5))                                                                              # Line chart size supports hour-by-hour comparison.
sns.lineplot(data=gymdb_hourly_usage, x='session_hour', y='session_count', marker='o')                  # Hourly usage is plotted as an operating pattern curve.
plt.title('Gym DB Hourly Usage Pattern')                                                                # Chart title describes peak-time analysis.
plt.xlabel('Session Hour')                                                                              # x-axis label identifies check-in hour.
plt.ylabel('Session Count')                                                                             # y-axis label identifies session volume.
plt.xticks(range(0, 24, 2))                                                                             # Two-hour ticks keep the axis readable.
figure_outputs.append(save_current_plot('gymdb_hourly_usage_pattern.png'))                              # The hourly usage plot is saved.

plt.figure(figsize=(9, 5))                                                                              # Horizontal layout supports city/location names.
sns.barplot(data=gymdb_location_utilization, y='location', x='session_count', hue='gym_type', dodge=False) # Branch sessions are plotted by location and gym type.
plt.title('Gym DB Location Utilization')                                                                # Chart title describes branch demand.
plt.xlabel('Session Count')                                                                             # x-axis label identifies session volume.
plt.ylabel('Location')                                                                                  # y-axis label identifies branch city.
plt.legend(title='Gym Type')                                                                            # Legend explains branch category colors.
figure_outputs.append(save_current_plot('gymdb_location_utilization.png'))                              # The location utilization plot is saved.

plt.figure(figsize=(7, 5))                                                                              # Compact chart size works for three subscription plans.
sns.barplot(data=gymdb_subscription_summary, x='subscription_plan', y='user_count')                      # Subscription plan counts are plotted.
plt.title('Gym DB Subscription Plan Mix')                                                               # Chart title identifies the membership mix.
plt.xlabel('Subscription Plan')                                                                         # x-axis label identifies subscription tier.
plt.ylabel('User Count')                                                                                # y-axis label identifies number of members.
figure_outputs.append(save_current_plot('gymdb_subscription_plan_mix.png'))                             # The subscription mix plot is saved.

plt.figure(figsize=(9, 5))                                                                              # Boxplot size supports comparison across workout categories.
sns.boxplot(data=gymdb_fact_sessions, x='workout_type', y='calories_per_minute')                        # Calories per minute compares workout intensity normalized by duration.
plt.title('Gym DB Calories per Minute by Workout Type')                                                 # Chart title links operational sessions to intensity context.
plt.xlabel('Workout Type')                                                                              # x-axis label identifies workout category.
plt.ylabel('Calories per Minute')                                                                       # y-axis label identifies normalized calorie burn.
plt.xticks(rotation=30, ha='right')                                                                     # Rotated labels prevent overlap.
figure_outputs.append(save_current_plot('gymdb_calories_per_minute_by_workout.png'))                    # The calories-per-minute plot is saved.

figure_manifest = pd.DataFrame({'figure_path': [str(path.relative_to(PROJECT_ROOT)) for path in figure_outputs]})                 # Figure paths are stored in a manifest table.
display(figure_manifest)                                                                                # The figure manifest confirms all PNG outputs.
figure_manifest_output = export_dataframe(figure_manifest, 'week1_figure_manifest.csv')                 # Figure manifest is exported for final-product handoff.


## Cell 18: Final Data Contract Validation

This cell checks that cleaned datasets contain the fields needed by final product modules.

The contracts protect Week 2 and Week 3 work from silent data-shape changes.
        

In [ ]:
required_columns = {                                                                                    # Each processed dataframe receives a minimum required-column contract.
    'sleep_clean': ['person_id', 'systolic_bp', 'diastolic_bp', 'readiness_score', 'readiness_band', 'archetype_cluster'], # Sleep fields needed for readiness and clustering.
    'gym_clean': ['bpm_response_ratio', 'intensity_band', 'calorie_burn_rate_kcal_per_hour', 'calories_burned_clean'], # Gym fields needed for Week 2 model training.
    'exercise_clean': ['exercise_id', 'name', 'body_parts', 'equipments', 'target_muscles', 'instruction_text'], # ExerciseDB fields needed for recommender and RAG.
    'nutrition_clean': ['food_item', 'meal_type', 'calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g', 'macro_total_g'], # Nutrition fields needed for macro dashboard.
    'gymdb_users_clean': ['user_id', 'age', 'age_group', 'gender', 'user_location', 'subscription_plan'], # Gym DB user fields needed for operational dashboard filters.
    'gymdb_fact_sessions': ['user_id', 'gym_id', 'checkin_time', 'checkout_time', 'duration_minutes', 'calories_per_minute', 'subscription_plan', 'gym_type', 'location'], # Gym DB fact fields needed for warehouse marts.
}                                                                                                       # End of required-column contracts.

contract_frames = {                                                                                     # Cleaned dataframes are mapped to contract names.
    'sleep_clean': sleep_clean,                                                                         # Sleep dataframe with readiness and clusters.
    'gym_clean': gym_clean,                                                                             # Gym dataframe with intensity and calorie target fields.
    'exercise_clean': exercise_clean,                                                                   # ExerciseDB dataframe with parsed list fields.
    'nutrition_clean': nutrition_clean,                                                                 # Nutrition dataframe with cleaned macro fields.
    'gymdb_users_clean': gymdb_users_clean,                                                             # Gym DB user dimension dataframe.
    'gymdb_fact_sessions': gymdb_fact_sessions,                                                         # Gym DB in-memory enriched session fact dataframe.
}                                                                                                       # End of dataframe mapping.

contract_rows = []                                                                                      # Contract results are collected into a final table.
for dataset_name, columns in required_columns.items():                                                  # Each dataset contract is checked independently.
    dataframe = contract_frames[dataset_name]                                                           # The cleaned dataframe is selected.
    missing_columns = [column for column in columns if column not in dataframe.columns]                 # Missing required columns indicate a broken handoff.
    contract_rows.append({                                                                              # One contract result is added per dataset.
        'dataset': dataset_name,                                                                        # Dataset contract name.
        'required_column_count': len(columns),                                                          # Number of required fields.
        'missing_required_columns': missing_columns,                                                    # Missing required columns if any.
        'contract_passed': len(missing_columns) == 0,                                                   # Boolean contract result.
    })                                                                                                  # End of one contract row.

contract_table = pd.DataFrame(contract_rows)                                                            # Contract results become a readable dataframe.
display(contract_table)                                                                                 # The data contract table is displayed before final manifest creation.
assert contract_table['contract_passed'].all(), 'One or more Week 1 data contracts failed.'             # The notebook stops if any required handoff field is missing.

contract_output = export_dataframe(contract_table, 'week1_data_contract_validation.csv')                # Data contract validation is exported for final-product evidence.


## Cell 19: Final TODO Checklist and Product Manifest

This cell produces the final Week 1 proof package.

The manifest records:

- Processed CSV files.
- Validation/proof files.
- Plot files.
- Summary JSON.
- TODO coverage.
- Final-product handoff destinations.
        

In [ ]:
todo_checklist = pd.DataFrame([                                                                         # A visible checklist maps notebook outputs back to TODO.md.
    {'todo_line': '691 + extension', 'todo_item': 'Load and inspect original 4 CSVs plus 4 Gym DB tables', 'status': 'complete', 'evidence': 'week1_raw_inspection_summary.csv + gymdb_raw_inspection_summary.csv'}, # Raw inspection evidence.
    {'todo_line': 692, 'todo_item': 'Clean Sleep dataset', 'status': 'complete', 'evidence': 'sleep_cleaning_proof.csv'}, # Sleep cleaning evidence.
    {'todo_line': 693, 'todo_item': 'Clean Gym Members dataset', 'status': 'complete', 'evidence': 'gym_cleaned_intensity_calories.csv'}, # Gym cleaning evidence.
    {'todo_line': 694, 'todo_item': 'Clean ExerciseDB string-list fields', 'status': 'complete', 'evidence': 'exercisedb_parse_proof.csv'}, # ExerciseDB parsing evidence.
    {'todo_line': 695, 'todo_item': 'Clean Nutrition macro columns', 'status': 'complete', 'evidence': 'nutrition_numeric_validation.csv'}, # Nutrition cleaning evidence.
    {'todo_line': 696, 'todo_item': 'Derive Training Readiness rule', 'status': 'complete', 'evidence': 'readiness_component_summary.csv'}, # Readiness rule evidence.
    {'todo_line': 697, 'todo_item': 'Outlier check on Calories_Burned', 'status': 'complete', 'evidence': 'gym_calorie_outlier_proof.csv'}, # Outlier proof evidence.
    {'todo_line': 700, 'todo_item': 'K-Means clustering on Sleep dataset', 'status': 'complete', 'evidence': 'sleep_kmeans_silhouette_scores.csv + sleep_cluster_profile.csv'}, # K-Means evidence.
    {'todo_line': 701, 'todo_item': 'Cross-dataset EDA', 'status': 'complete', 'evidence': 'week1_dashboard_summary_tables.json'}, # Cross-dataset EDA evidence.
    {'todo_line': 702, 'todo_item': 'Save EDA plots as PNG', 'status': 'complete', 'evidence': 'week1_figure_manifest.csv'}, # Plot export evidence.
    {'todo_line': 'extension', 'todo_item': 'Integrate Gym Database Management System for post-Week-1 warehouse/dashboard work', 'status': 'complete', 'evidence': 'gymdb_relationship_validation.csv + gymdb_mart_*.csv'}, # Gym DB extension evidence.
])                                                                                                      # End of final TODO checklist.

processed_outputs = [                                                                                   # All important Week 1 processed and proof files are listed for the manifest.
    raw_inspection_output, gymdb_raw_inspection_output, nutrition_repair_summary_output, sleep_cleaning_proof_output, # Raw and cleaning proof outputs.
    readiness_component_output, sleep_readiness_output, gym_output, outlier_proof_output,               # Readiness and gym outputs.
    exercise_output, exercise_parse_proof_output, nutrition_output, nutrition_validation_output,        # Exercise and nutrition outputs.
    sleep_clusters_output, silhouette_output, cluster_profile_output, cluster_readiness_output,         # Clustering outputs.
    gymdb_users_output, gymdb_plans_output, gymdb_locations_output, gymdb_table_cleaning_proof_output, # Gym DB dimension and cleaning proof outputs.
    gymdb_fact_sample_output, gymdb_relationship_output, gymdb_warehouse_output,                       # Gym DB fact sample and warehouse proof outputs.
    gymdb_session_by_workout_output, gymdb_subscription_output, gymdb_location_output,                 # Gym DB dashboard mart outputs.
    gymdb_monthly_output, gymdb_hourly_output, gymdb_gym_type_output,                                  # Gym DB time and gym-type mart outputs.
    figure_manifest_output, contract_output,                                                            # Figure and contract outputs.
]                                                                                                       # End of processed output list.

processed_file_manifest = []                                                                            # Reloaded file metadata is collected for validation.
for output_path in processed_outputs:                                                                   # Each exported CSV is reloaded for proof of readability.
    exported_df = pd.read_csv(output_path)                                                              # Reloading checks the saved file rather than only memory state.
    processed_file_manifest.append({                                                                    # One metadata record is stored per exported file.
        'path': str(output_path.relative_to(PROJECT_ROOT)),                                             # Repository-relative output path.
        'rows': int(exported_df.shape[0]),                                                              # Exported row count.
        'columns': int(exported_df.shape[1]),                                                           # Exported column count.
        'missing_values': int(exported_df.isna().sum().sum()),                                          # Total missing values after export.
    })                                                                                                  # End of one output metadata record.

todo_checklist_output = export_dataframe(todo_checklist, 'week1_todo_completion_checklist.csv')         # TODO checklist is exported.

final_manifest = {                                                                                      # The final manifest becomes the top-level Week 1 handoff document.
    'project': 'Smart Workout DSS/BIS',                                                                 # Project name.
    'scope': 'Week 1 Earth and Non data foundation complete package with Gym DB warehouse extension',    # Manifest scope.
    'project_root': str(PROJECT_ROOT),                                                                  # Project root used by the notebook.
    'processed_dir': str(PROCESSED_DIR.relative_to(PROJECT_ROOT)),                                      # Processed output folder.
    'figures_dir': str(FIGURES_DIR.relative_to(PROJECT_ROOT)),                                          # Figure output folder.
    'summary_tables_path': str(summary_tables_path.relative_to(PROJECT_ROOT)),                          # Dashboard summary JSON path.
    'selected_sleep_cluster_k': selected_k,                                                             # Selected K-Means cluster count.
    'gymdb_full_fact_exported': SAVE_FULL_GYMDB_FACT,                                                   # Full fact export decision is documented.
    'processed_files': processed_file_manifest,                                                         # All processed/proof CSV metadata.
    'figure_files': [str(path.relative_to(PROJECT_ROOT)) for path in figure_outputs],                   # All saved PNG figure paths.
    'todo_checklist': todo_checklist.to_dict(orient='records'),                                         # TODO completion records.
    'final_product_handoff': {                                                                          # Downstream usage map.
        'week2_ml': ['gym_cleaned_intensity_calories.csv', 'sleep_cleaned_readiness_clusters.csv'],     # Files for Week 2 model training.
        'week3_dashboard': ['week1_dashboard_summary_tables.json', 'docs/figures/week1/*.png', 'gymdb_mart_*.csv'], # Files for Week 3 dashboard views.
        'warehouse_bi': ['gymdb_dim_users_cleaned.csv', 'gymdb_dim_locations_cleaned.csv', 'gymdb_dim_subscription_plans_cleaned.csv', 'gymdb_mart_*.csv'], # Files for Gym DB BI layer.
        'recommender_rag': ['exercisedb_cleaned_catalog.csv'],                                          # Exercise catalogue for recommender and RAG.
        'report_evidence': ['week1_manifest_complete.json', 'week1_todo_completion_checklist.csv', 'gymdb_relationship_validation.csv'], # Files for report evidence.
    },                                                                                                  # End of downstream usage map.
}                                                                                                       # End of final manifest dictionary.

final_manifest_path = PROCESSED_DIR / 'week1_manifest_complete.json'                                    # The final manifest is saved beside processed outputs.
final_manifest_path.write_text(json.dumps(final_manifest, indent=2), encoding='utf-8')                  # Readable JSON supports manual and programmatic review.

display(todo_checklist)                                                                                 # Final TODO checklist is displayed.
display(pd.DataFrame(processed_file_manifest))                                                          # Processed file manifest is displayed.
print('Final manifest saved:', final_manifest_path)                                                     # The final manifest path is printed.
print('Week 1 complete package with Gym DB extension is ready for final-product handoff.')               # Completion message confirms the notebook reached the end.


## Final Product Handoff Summary

After running this notebook from the project root, the following folders should contain the Week 1 complete package:

```text
data/processed/
  week1_raw_inspection_summary.csv
  sleep_cleaning_proof.csv
  sleep_cleaned_readiness.csv
  sleep_cleaned_readiness_clusters.csv
  gym_cleaned_intensity_calories.csv
  gym_calorie_outlier_proof.csv
  exercisedb_cleaned_catalog.csv
  exercisedb_parse_proof.csv
  nutrition_cleaned_macros.csv
  nutrition_csv_repair_log.csv
  nutrition_numeric_validation.csv
  sleep_kmeans_silhouette_scores.csv
  sleep_cluster_profile.csv
  sleep_cluster_readiness_mix.csv
  week1_dashboard_summary_tables.json
  week1_figure_manifest.csv
  week1_data_contract_validation.csv
  week1_todo_completion_checklist.csv
  week1_manifest_complete.json

docs/figures/week1/
  workout_type_distribution.png
  calories_by_experience_level.png
  intensity_band_distribution.png
  exercise_coverage_by_body_part.png
  exercise_coverage_by_equipment.png
  nutrition_macro_mix_by_meal_type.png
  training_readiness_distribution.png
  sleep_archetype_clusters.png
```

This output is enough for the Week 1: data cleaning, validation proof, descriptive analytics, K-Means profiles, EDA assets, and final-product handoff.
        
